# 39 - Replicate L1 vs triplet with a new training seed

Run all cells in Colab with a GPU, after notebook 00 if the runtime was reset.
Then run notebook 40. Only `native_l1` and `native_triplet` are trained.

Both arms start from the original 23 warmstart, with the same two-epoch budget,
task order, losses and frozen checkpoint selector as 37. Training corruptions
use seed 20261127 for both arms. Original evaluation and provenance seeds stay
fixed. The checksummed TRAIN-only bank from 37 is shared, and new checkpoints
and the completion pointer are written to `experiments/native_triplet_replication_v1`.
No independent or external test is used.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import shutil
import sys
import time
import types
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from google.colab import drive
DEVICE = torch.device('cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M1_DIR = PROJECT_ROOT / 'experiments' / 'm1_shared_curriculum_v1_seed20261110' / 'dd1427cfad6621c577bf'
SEED = 20261110
REPLICATION_SEED = 20261127
RUN_SEEDS = {'dry_l1_seed20261110': 20261110}
RUN_NAMES = tuple(RUN_SEEDS)
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
SPECIALIZATION_ROOT = PROJECT_ROOT / 'experiments' / 'm5_dry_specialization_v1'
LAMBDA_GRADIENT = 0.1
IDENTITY_FRACTION = 0.1

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact changed: {directory / name}')
print('Read-only prerequisites on CPU; DMD inference device selected below.')
MATCHER_ROOT = PROJECT_ROOT / 'experiments' / 'risk_patch_matcher_v1'
MATCHER_SETTINGS = {'epochs': 6, 'steps_per_epoch': 200, 'identities_per_batch': 12,
    'learning_rate': 0.0002, 'embedding_dim': 128, 'held_groups_per_db': 8,
    'quality_gate': {'clean_auc': 0.80, 'dry_auc': 0.70, 'clean_tar': 0.25, 'dry_tar': 0.10}}

import ast
import io
import sys
import subprocess
import urllib.request
import zipfile
from scipy.optimize import linear_sum_assignment
import scipy
import torch.nn.functional as F

INFERENCE_DEVICE = torch.device('cpu')
SOURCE33_DIR = PROJECT_ROOT / 'diagnostics' / 'pretrained_dmd_feasibility_v1' / '411576c19ecb440b8392'
print('34: cached local scalar and one live input-gradient check on CPU; no training/GPU/NBIS.')

SOURCE34_DIR = PROJECT_ROOT/'diagnostics'/'dmd_local_surrogate_v1'/'af30fd0800005636bbb8'
CHECK_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHUNK_SIZE = 2
print('35 frozen matching; live check device:',CHECK_DEVICE,'| chunk size:',CHUNK_SIZE,'| NO TRAINING')

SOURCE35_DIR=PROJECT_ROOT/'diagnostics'/'frozen_before_lsar_check_v1'/'29637964d1e0f12d2ad8'
EXPECTED35_SHA='29637964d1e0f12d2ad87089ed12ff2ac539354a2f5ec395450a97092457c86c'
print('36: same frozen score; FULL native-image restoration; no training.')

import io
import platform
import subprocess
import zipfile
from pathlib import PurePosixPath
if not torch.cuda.is_available():
    raise RuntimeError('37 trains: select a Colab GPU. T4 first; do not silently run this pilot on CPU.')
CHECK_DEVICE=DEVICE=torch.device('cuda')
PILOT_RUNS=('native_l1','native_triplet')
PILOT_EPOCHS=2
EFFECTIVE_BATCH=8
CHECKPOINT_EVERY=25


In [ ]:
print('36 prerequisite: protocols',flush=True)
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_config = json.loads((M1_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_completion = json.loads((M1_DIR / 'completion.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
m0_sha, m1_sha = (file_digest(directory / 'configuration.json') for directory in (M0_DIR, M1_DIR))
verify_files(M0_DIR, m0_config['source_file_sha256'])
verify_files(M1_DIR, m1_config['source_file_sha256'])
verify_files(M1_DIR, m1_completion['file_sha256'])
if (m1_completion['status'] != 'shared_curriculum_control_complete' or m1_completion['configuration_sha256'] != m1_sha
        or m1_config['m0_configuration_sha256'] != m0_sha):
    raise RuntimeError('Frozen completed M1 provenance mismatch.')
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (m0_config, sim_config, pair_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or m1_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']):
    raise RuntimeError('Frozen simulator/normalization provenance mismatch.')
for config in (m0_config, m1_config):
    if (str(torch.__version__) != config['torch_version'] or str(torch.version.cuda) != config['torch_cuda']
            or np.__version__ != config['numpy_version'] or Image.__version__ != config['pillow_version']):
        raise RuntimeError('Library versions differ from frozen training.')
if Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:
    raise RuntimeError('Normalization/simulator library versions differ.')
for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv'):
    if file_digest(M1_DIR / name) != file_digest(M0_DIR / name):
        raise RuntimeError(f'M1 model/source differs from M0: {name}')
SETTINGS = dict(m1_config['settings'])
EPOCHS = len(m1_config['epoch_conditions'])
if (EPOCHS != sum(SETTINGS['phase_lengths']) or EPOCHS != m0_config['epochs']
        or SETTINGS['base_channels'] != m0_config['base_channels'] or SETTINGS['patch_size'] != m0_config['patch_size']
        or SETTINGS['batch_size'] != m0_config['batch_size'] or SETTINGS['learning_rate'] != m0_config['learning_rate']):
    raise RuntimeError('Budget/architecture/optimizer differ from the frozen baseline.')
train_sources = pd.read_csv(M0_DIR / 'train_sources.csv', keep_default_na=False)
validation_sources = pd.read_csv(M0_DIR / 'validation_sources.csv', keep_default_na=False)
for split_name, frame in (('train', train_sources), ('validation', validation_sources)):
    allowed = locked[locked['split'].eq(split_name)].set_index('relative_path')
    if frame.empty or frame['relative_path'].duplicated().any() or frame.groupby(['dataset', 'db']).ngroups != 6 or set(frame['dataset']) != {'FVC2000', 'FVC2002'}:
        raise RuntimeError(f'Invalid source-only {split_name} data.')
    for record in frame.to_dict('records'):
        relative = record['relative_path']
        if (relative not in allowed.index or allowed.loc[relative, 'sha256'] != record['sha256']
                or allowed.loc[relative, 'identity_key'] != record['identity_key']):
            raise RuntimeError(f'Split/source mismatch: {relative}')
        path = DATA_ROOT / relative
        if split_name == 'train' and (not path.is_file() or file_digest(path) != record['sha256']):
            raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02.')
if (set(train_sources['identity_key']) & set(validation_sources['identity_key'])
        or set(train_sources['sha256']) & set(validation_sources['sha256'])
        or not validation_sources.groupby(['dataset', 'db']).size().eq(12).all()):
    raise RuntimeError('Train/validation overlap or changed validation coverage.')
training = load_module('m2_original_model_v1', M0_DIR / 'restoration_baseline.py')
simulator = load_module('m2_original_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
curriculum_data = load_module('m2_frozen_task_data_v1', M1_DIR / 'curriculum_data.py')
FROZEN_ENGINE_SOURCE = (M1_DIR / 'training_helpers.py.txt').read_bytes().decode('utf-8')
SOURCE_COUNT = len(train_sources)
all_conditions = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
print('Verified unchanged sources:', SOURCE_COUNT, 'train /', len(validation_sources), 'validation.')

allowed_train_paths = frozenset(train_sources.relative_path)


In [ ]:
print('36 prerequisite: load',flush=True)

pointer = json.loads((MATCHER_ROOT / 'completed_experiment.json').read_text(encoding='utf-8'))
MATCHER_DIR = Path(pointer['directory'])
if (MATCHER_DIR.parent != MATCHER_ROOT
        or file_digest(MATCHER_DIR / 'configuration.json') != pointer['configuration_sha256']
        or file_digest(MATCHER_DIR / 'completion.json') != pointer['completion_sha256']):
    raise RuntimeError('Notebook29 published pointer/integrity mismatch.')
configuration = json.loads((MATCHER_DIR / 'configuration.json').read_text(encoding='utf-8'))
completion = json.loads((MATCHER_DIR / 'completion.json').read_text(encoding='utf-8'))
if (configuration['version'] != 'risk_patch_matcher_v1'
        or configuration['settings'] != MATCHER_SETTINGS or configuration['seed'] != SEED
        or completion['status'] != 'risk_matcher_fit_complete'
        or completion['configuration_sha256'] != pointer['configuration_sha256']
        or completion['completed_epochs'] != MATCHER_SETTINGS['epochs']
        or configuration['selection'] != 'last fixed-budget checkpoint; no gate/validation epoch selection'
        or configuration['train_sources_sha256'] != file_digest(M0_DIR / 'train_sources.csv')
        or configuration['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json')
        or configuration['simulator_sha256'] != file_digest(SIM_DIR / 'fingerprint_degradation.py')
        or configuration['patch_size'] != SETTINGS['patch_size']):
    raise RuntimeError('Expected unchanged completed29 fixed-budget matcher configuration.')
verify_files(MATCHER_DIR, completion['file_sha256'])
for key, actual in (('torch_version', str(torch.__version__)), ('torch_cuda', str(torch.version.cuda)),
        ('numpy_version', np.__version__), ('pillow_version', Image.__version__)):
    if configuration[key] != actual:
        raise RuntimeError(f'Matcher library version differs: {key}')
matcher_source = (MATCHER_DIR / 'matcher_source.py').read_bytes().decode('utf-8')
if (hashlib.sha256(matcher_source.encode()).hexdigest() != configuration['model_source_sha256']
        or file_digest(MATCHER_DIR / 'inner_partition.csv') != configuration['partition_sha256']
        or file_digest(MATCHER_DIR / 'checkpoint_helpers.py.txt') != configuration['checkpoint_source_sha256']):
    raise RuntimeError('Frozen matcher source/partition hash differs.')
matcher = types.ModuleType('read_only_failed29_matcher')
matcher.__dict__.update(globals())
exec(compile(matcher_source, str(MATCHER_DIR / 'matcher_source.py'), 'exec'), matcher.__dict__)
partition = pd.read_csv(MATCHER_DIR / 'inner_partition.csv', keep_default_na=False)
expected_partition = matcher.inner_partition(train_sources, MATCHER_SETTINGS['held_groups_per_db'])
pd.testing.assert_frame_equal(partition, expected_partition, check_dtype=False)
latest = json.loads((MATCHER_DIR / 'latest.json').read_text(encoding='utf-8'))
if (latest != completion['checkpoint'] or Path(latest['file']).name != latest['file']
        or latest['configuration_sha256'] != pointer['configuration_sha256']
        or latest['completed_epochs'] != MATCHER_SETTINGS['epochs']
        or file_digest(MATCHER_DIR / latest['file']) != latest['sha256']):
    raise RuntimeError('LAST completed29 checkpoint differs.')
state = torch.load(MATCHER_DIR / latest['file'], map_location='cpu', weights_only=True)
if (state['configuration_sha256'] != pointer['configuration_sha256']
        or state['completed_epochs'] != MATCHER_SETTINGS['epochs']):
    raise RuntimeError('Saved matcher payload differs.')
with torch.random.fork_rng(devices=[]):
    model = matcher.PatchMatcher().to(DEVICE)
model.load_state_dict(state['model'], strict=True)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
if any(not torch.isfinite(value).all() for value in model.state_dict().values()):
    raise RuntimeError('Nonfinite frozen matcher state.')
saved_state = {key: value.clone() for key, value in model.state_dict().items()}
saved_history = pd.DataFrame(state['history'])
if saved_history.epoch.tolist() != list(range(1, MATCHER_SETTINGS['epochs'] + 1)):
    raise RuntimeError('Matcher has not trained the full predefined budget.')
del state
calibration = pd.read_csv(MATCHER_DIR / 'calibration_scores.csv', float_precision='round_trip')
gate = pd.read_csv(MATCHER_DIR / 'gate_scores.csv', float_precision='round_trip')
saved_gate = json.loads((MATCHER_DIR / 'quality_gate.json').read_text(encoding='utf-8'))
saved_thresholds = json.loads((MATCHER_DIR / 'thresholds.json').read_text(encoding='utf-8'))
if saved_gate != completion['quality_gate']:
    raise RuntimeError('Saved29 gate attestation differs.')
print('Read-only loaded matcher epoch:', latest['completed_epochs'], '| original gate passed:', saved_gate['passed'])


In [ ]:
print('36 prerequisite: helpers',flush=True)
AUDIT_SOURCE = 'def panel_records(partition, split):\n    if split not in (\'fit\', \'gate\', \'calibration\'):\n        raise ValueError(\'Only explicit main-TRAIN inner partitions are supported.\')\n    selected = partition[partition.matcher_split.eq(split)]\n    if selected.empty:\n        raise RuntimeError(\'Missing inner partition.\')\n    if split == \'fit\':\n        frames = []\n        for (dataset, db), rows in selected.groupby([\'dataset\', \'db\'], sort=True):\n            groups = sorted(rows.connected_group.unique(),\n                key=lambda key: training.stable_seed(f\'{SEED}|audit_fit_panel|{key}\'))\n            if len(groups) < MATCHER_SETTINGS[\'held_groups_per_db\']:\n                raise RuntimeError(\'Insufficient fit groups for matched audit panel.\')\n            frames.append(rows[rows.connected_group.isin(groups[:MATCHER_SETTINGS[\'held_groups_per_db\']])])\n        selected = pd.concat(frames, ignore_index=True)\n    galleries, probes = [], []\n    for key, rows in selected.groupby(\'identity_key\', sort=True):\n        records = rows.sort_values(\'relative_path\').drop_duplicates(\'sha256\').to_dict(\'records\')\n        if len(records) < 3:\n            raise RuntimeError(f\'Audit requires the same three byte-distinct source impressions: {key}\')\n        galleries.append(records[0])\n        probes.extend(records[1:3])\n    return galleries, probes\n\ndef pair_metadata(partition, split):\n    galleries, probes = panel_records(partition, split)\n    rows = []\n    for condition in (\'identity/none\', \'dry_like/mild\', \'dry_like/moderate\', \'dry_like/severe\'):\n        for probe in probes:\n            for gallery in galleries:\n                if (probe[\'dataset\'], probe[\'db\']) != (gallery[\'dataset\'], gallery[\'db\']):\n                    continue\n                label = int(probe[\'identity_key\'] == gallery[\'identity_key\'])\n                if not label and probe[\'connected_group\'] == gallery[\'connected_group\']:\n                    continue\n                rows.append(dict(dataset=probe[\'dataset\'], db=probe[\'db\'], condition=condition,\n                    probe=probe[\'relative_path\'], gallery=gallery[\'relative_path\'], label=label))\n    return pd.DataFrame(rows)\n\ndef verify_score_metadata(frame, partition, split):\n    keys = [\'dataset\', \'db\', \'condition\', \'probe\', \'gallery\', \'label\']\n    if (frame.empty or frame.duplicated(keys).any() or not np.isfinite(frame.score).all()\n            or not frame.score.between(-1.00001, 1.00001).all() or set(frame.label) != {0, 1}):\n        raise RuntimeError(\'Invalid saved inner verification scores.\')\n    expected = pair_metadata(partition, split)\n    pd.testing.assert_frame_equal(frame[keys].sort_values(keys).reset_index(drop=True),\n        expected[keys].sort_values(keys).reset_index(drop=True), check_dtype=False)\n\ndef crop_details(record):\n    relative = record[\'relative_path\']\n    if relative not in allowed_train_paths:\n        raise RuntimeError(\'Crop audit can read main TRAIN images only.\')\n    dpi = pair_config[\'source_dpi\'][f"{record[\'dataset\']}_{record[\'db\']}"]\n    reference = training.normalized_image(str(DATA_ROOT / relative), dpi)\n    foreground = simulator.foreground_mask(reference)\n    ys, xs = np.where(foreground)\n    if not len(ys):\n        raise RuntimeError(f\'No foreground for frozen crop: {relative}\')\n    crop_seed = training.stable_seed(f\'{SEED}|0|{relative}|crop\')\n    rng = np.random.default_rng(crop_seed)\n    anchor = int(rng.integers(len(ys)))\n    size = SETTINGS[\'patch_size\']\n    height, width = reference.shape\n    top = int(np.clip(int(ys[anchor]) - size // 2, 0, max(0, height - size)))\n    left = int(np.clip(int(xs[anchor]) - size // 2, 0, max(0, width - size)))\n    pad = ((0, max(0, size - height)), (0, max(0, size - width)))\n    padded = np.pad(reference, pad, mode=\'reflect\') if any(value for pair in pad for value in pair) else reference\n    patch = np.array(padded[top:top + size, left:left + size], dtype=np.float32, copy=True)\n    actual = matcher.sample_patches([record], 0, [(\'identity\', \'none\')])[0, 0].numpy()\n    if not np.array_equal(patch, actual):\n        raise RuntimeError(\'Reconstructed crop is not byte-identical to frozen29 sampling.\')\n    coverage = (min(size, height - top) * min(size, width - left)) / (height * width)\n    metadata = dict(relative_path=relative, identity_key=record[\'identity_key\'], dataset=record[\'dataset\'],\n        db=record[\'db\'], crop_seed=crop_seed, normalized_height=height, normalized_width=width,\n        top=top, left=left, patch_size=size, padded_bottom=pad[0][1], padded_right=pad[1][1],\n        crop_image_fraction=coverage, crop_foreground_fraction=float(\n            foreground[top:min(top + size, height), left:min(left + size, width)].mean()),\n        crop_sha256=hashlib.sha256(patch.astype(\'<f4\').tobytes()).hexdigest())\n    return metadata, reference, patch\n\ndef describe_scores(values):\n    values = np.asarray(values, dtype=np.float64)\n    if not len(values) or not np.isfinite(values).all():\n        raise RuntimeError(\'Nonempty finite scores required for descriptive summaries.\')\n    return dict(count=len(values), mean=float(values.mean()), std=float(values.std(ddof=1)) if len(values) > 1 else None,\n        q05=float(np.quantile(values, .05)), median=float(np.median(values)), q95=float(np.quantile(values, .95)))\n\ndef auc_against(genuine, impostors):\n    genuine, impostors = np.asarray(genuine), np.asarray(impostors)\n    if not len(genuine) or not len(impostors):\n        raise RuntimeError(\'Both positive and negative scores required.\')\n    return float(np.mean((genuine[:, None] > impostors).astype(float)\n        + .5 * (genuine[:, None] == impostors)))\n\n@torch.inference_mode()\ndef embed_panel(records, condition):\n    features = []\n    for start in range(0, len(records), 12):\n        images = matcher.sample_patches(records[start:start + 12], 0, [condition]).to(DEVICE)\n        value = model(images)\n        if not torch.isfinite(value).all():\n            raise RuntimeError(\'Nonfinite CPU embeddings.\')\n        features.append(value.cpu())\n    return torch.cat(features)\n\ndef summarize_panel(frame):\n    summaries = []\n    for (split, dataset, db, condition), group in frame.groupby(\n            [\'inner_split\', \'dataset\', \'db\', \'condition\'], sort=True):\n        threshold = saved_thresholds[f\'{dataset}_{db}\']\n        positive = group[group.pool.eq(\'different_impression_genuine\')].score.to_numpy()\n        for pool in (\'different_impression_genuine\', \'same_source_control\', \'same_db_impostor\', \'cross_db_impostor\'):\n            values = group[group.pool.eq(pool)].score.to_numpy()\n            summaries.append(dict(inner_split=split, dataset=dataset, db=db, condition=condition, pool=pool,\n                threshold=threshold, acceptance_fraction=float(np.mean(values >= threshold)),\n                auc_against_cross_impression_genuine=auc_against(positive, values)\n                    if pool.endswith(\'impostor\') else None, **describe_scores(values)))\n    return pd.DataFrame(summaries)\n'
exec(compile(AUDIT_SOURCE, '<matcher-failure-audit>', 'exec'))


In [ ]:
print('36 prerequisite: replay',flush=True)

verify_score_metadata(calibration, partition, 'calibration')
verify_score_metadata(gate, partition, 'gate')
thresholds, replay_metrics, replay_summary = matcher.gate_analysis(calibration, gate)
if saved_thresholds != {f'{dataset}_{db}': value for (dataset, db), value in thresholds.items()}:
    raise RuntimeError('Exact CSV replay changed original calibration thresholds.')
if replay_summary != saved_gate:
    raise RuntimeError('Saved gate summary differs from exact saved-score replay.')
prior_metrics = pd.read_csv(MATCHER_DIR / 'gate_metrics.csv', float_precision='round_trip')
pd.testing.assert_frame_equal(replay_metrics, prior_metrics, check_dtype=False, check_exact=True)
print('Exact saved-score replay verified; gate UNCHANGED:')
print(json.dumps(replay_summary, indent=2))
print('Saved fixed-budget learning/runtime/memory history:')
print(saved_history.to_string(index=False))


In [ ]:
print('36 prerequisite: source33',flush=True)

EXPECTED_DMD_FILES = {'models/model_zoo.py': '39f24735c03bcbbc0d51f6af75146bd3beb55a17bc25cf1a3b1c8f99bdd54b29', 'models/resnet.py': 'e6d28ed8a9730fc0ce5f1a98d464a129cd976ad91028e4516880908b42688e8d', 'models/units.py': '3c685be67e6ad57c5016cf44f75aa7660ba4675f17277faec470d44b3215a1aa', 'models/inception.py': 'bac60ed9f06c9e2e0869f73bb2bf987bbd3889856e4509f9a0cec5070d05702d', 'models/dataloader_densemnt.py': '70308d0b9a6258f5fb2eb815fa296a987190c0db8d9e5750c00c9df16e9468aa', 'evaluate_mnt.py': '84e7f4e62ebdaf718bc1266367cee5fa7680fd599d585c3cc9398f35a4e5b78f', 'README.md': '2b538fe64ef3510f60512a7b413eb04bbfc66a4c51b4fa00aa67e533eb84ee68', 'LICENCE': '4878c4e24ae4dd9878852724b4ae3313dc9a0599680d6552c1606bbc3d63830c', 'DMD.yaml': '12186bc0a33ab479d9c041aae8e35a180a286b98500c3d990475403732538b3d'}
EXPECTED_DMD_COMMIT = 'feb0ad40e5f340caebb457237d7ef21d5d5724e6'
EXPECTED_WEIGHT_SHA = '6d89dff9e5502cf4534b0fd9e014b3fb4a494a0be904015319c72edd9a7aff48'
EXPECTED_ADAPTER_SHA = '82cde5da161f7c37b58fe1ce069b460965c2a94eb175d12405840fffdc6c0919'
source_config = json.loads((SOURCE33_DIR / 'configuration.json').read_text(encoding='utf-8'))
source33_sha = file_digest(SOURCE33_DIR / 'configuration.json')
source_completion = json.loads((SOURCE33_DIR / 'completion.json').read_text(encoding='utf-8'))
if (source_completion['status'] != 'pretrained_dmd_feasibility_complete'
        or source_completion['configuration_sha256'] != source33_sha
        or SOURCE33_DIR.name != source33_sha[:20]
        or source_config['version'] != 'pretrained_dmd_feasibility_v1'
        or source_config['source_commit'] != EXPECTED_DMD_COMMIT
        or source_config['source_files'] != EXPECTED_DMD_FILES
        or source_config['weight_sha256'] != EXPECTED_WEIGHT_SHA
        or source_config['weight_bytes'] != 179093287
        or source_config['max_anchors'] != 24 or source_config['minimum_anchors'] != 4
        or source_config['adapter_sha256'] != EXPECTED_ADAPTER_SHA
        or source_config['matcher29_configuration_sha256'] != pointer['configuration_sha256']
        or source_config['matcher29_completion_sha256'] != pointer['completion_sha256']
        or source_completion['permits_restoration_training'] is not False):
    raise RuntimeError('Expected the unchanged completed33 source, not a different run/checkpoint.')
verify_files(SOURCE33_DIR,source_completion['file_sha256'])
for name,value in [('torch_version',str(torch.__version__)),('torch_cuda',str(torch.version.cuda)),
        ('numpy_version',np.__version__),('pillow_version',Image.__version__),('scipy_version',scipy.__version__)]:
    if source_config[name] != value:
        raise RuntimeError(f'Frozen33 library differs: {name}')
ASSET_ROOT = PROJECT_ROOT / 'pretrained' / 'dmd_sd14_v1' / EXPECTED_DMD_COMMIT
verify_files(ASSET_ROOT,EXPECTED_DMD_FILES)
WEIGHT_PATH = ASSET_ROOT / 'best_model.pth.tar'
if file_digest(WEIGHT_PATH) != EXPECTED_WEIGHT_SHA or WEIGHT_PATH.stat().st_size != 179093287:
    raise RuntimeError('Verified public DMD weights missing/changed; no unsafe fallback.')
adapter33_source = (SOURCE33_DIR / 'adapter.py.txt').read_bytes().decode('utf-8')
if hashlib.sha256(adapter33_source.encode()).hexdigest() != EXPECTED_ADAPTER_SHA:
    raise RuntimeError('Frozen33 adapter changed.')
exec(compile(adapter33_source,'<unchanged33-adapter>','exec'))
local_score,_unused_lsar = load_scoring(ASSET_ROOT / 'evaluate_mnt.py')
del _unused_lsar
dmd_model = load_dmd(ASSET_ROOT,WEIGHT_PATH)
initial_dmd_sha = state_sha(dmd_model)
print('Verified and loaded frozen33:',SOURCE33_DIR.name,'| previous LSA-R gate:',source_completion['quality']['passed'])


In [ ]:
print('36 prerequisite: local-scalar',flush=True)
LOCAL_SOURCE = "def query_anchor_index(relative, count, epoch=0):\n    if count < 4:\n        raise RuntimeError('Local surrogate needs at least4 original anchors, same as33.')\n    return training.stable_seed(f'{SEED}|{epoch}|{relative}|dmd_anchor_v1') % count\n\ndef read_cached33(record, family, severity):\n    relative = record['relative_path']\n    if relative not in allowed_train_paths:\n        raise RuntimeError('Cached DMD local analysis is main TRAIN only.')\n    directory = SOURCE33_DIR / 'anchors' / hashlib.sha256(relative.encode()).hexdigest()[:24]\n    expected = dict(relative_path=relative,source_sha256=record['sha256'],\n        configuration_sha256=source33_sha,mindtct_sha256=source_config['nbis_build']['binary_sha256']['mindtct'])\n    if cached_bundle(directory,expected) is None:\n        raise RuntimeError(f'Missing completed33 anchor cache: {relative}. Run33; do not silently re-extract.')\n    with np.load(directory / 'anchors.npz',allow_pickle=False) as archive:\n        anchors = archive['anchors'].copy()\n        shape = archive['image_shape'].copy()\n    parsed = parse_nbis_xyt((directory / 'minutiae.xyt').read_text(encoding='ascii'),*shape)\n    if not np.array_equal(anchors,parsed):\n        raise RuntimeError('Frozen anchors disagree with their XYT source.')\n    if len(anchors) < 4:\n        return dict(status='unmatchable_original_anchor_count',anchors=anchors,feature=None,mask=None)\n    feature_key = hashlib.sha256(f'{relative}|{family}|{severity}'.encode()).hexdigest()[:24]\n    directory = SOURCE33_DIR / 'features' / feature_key\n    expected = dict(relative_path=relative,family=family,severity=severity,\n        configuration_sha256=source33_sha,anchors_sha256=hashlib.sha256(anchors.tobytes()).hexdigest())\n    if cached_bundle(directory,expected) is None:\n        raise RuntimeError(f'Missing completed33 feature cache: {relative}/{family}/{severity}. Run33.')\n    with np.load(directory / 'features.npz',allow_pickle=False) as archive:\n        feature = torch.from_numpy(archive['feature'].copy())\n        mask = torch.from_numpy(archive['mask'].copy())\n    if (feature.shape != (len(anchors),768) or mask.shape != (len(anchors),64)\n            or not torch.isfinite(feature).all() or not torch.isfinite(mask).all()\n            or not ((mask>=0)&(mask<=1)).all()):\n        raise RuntimeError('Invalid cached33 local features.')\n    return dict(status='matched',anchors=anchors,feature=feature,mask=mask)\n\ndef local_identity_score(query_feature,query_mask,gallery_feature,gallery_mask):\n    if (query_feature.shape!=(1,768) or query_mask.shape!=(1,64)\n            or gallery_feature.ndim!=2 or gallery_feature.shape[1]!=768\n            or gallery_mask.shape!=(len(gallery_feature),64) or len(gallery_feature)<4\n            or len(gallery_feature)>24):\n        raise RuntimeError('Expected one local query and one4..24-anchor identity template.')\n    for value in (query_feature,query_mask,gallery_feature,gallery_mask):\n        if not torch.isfinite(value).all():\n            raise RuntimeError('Nonfinite local surrogate input.')\n    matrix = local_score(query_feature[None],gallery_feature[None],query_mask[None],gallery_mask[None],\n        ndim_feat=12,Normalize=False,binary=False)[0]\n    result = matrix.max(dim=1).values.squeeze(0)\n    if not torch.isfinite(result):\n        raise RuntimeError('Nonfinite local identity score.')\n    return result\n\ndef aligned_training_patch(image,anchor):\n    if image.ndim!=4 or image.shape[:2]!=(1,1) or not torch.isfinite(image).all():\n        raise RuntimeError('Expected one finite N1HW source.')\n    point = torch.as_tensor(anchor,device=image.device,dtype=image.dtype).flatten()\n    if len(point)<3 or not torch.isfinite(point).all():\n        raise RuntimeError('Expected a finite fixed original anchor.')\n    height,width = image.shape[-2:]\n    yy,xx = torch.meshgrid(torch.arange(192,device=image.device,dtype=image.dtype)-96,\n        torch.arange(192,device=image.device,dtype=image.dtype)-96,indexing='ij')\n    theta = torch.deg2rad(point[2])\n    sx = point[0]+torch.cos(theta)*xx-torch.sin(theta)*yy\n    sy = point[1]+torch.sin(theta)*xx+torch.cos(theta)*yy\n    grid = torch.stack(((2*sx+1)/width-1,(2*sy+1)/height-1),dim=-1)[None]\n    return F.grid_sample(image-.5,grid,mode='bilinear',padding_mode='zeros',align_corners=False)+.5\n\ndef local_prediction_features(prediction):\n    if (prediction.ndim!=4 or prediction.shape[1:]!=(1,192,192)\n            or not torch.isfinite(prediction).all()):\n        raise RuntimeError('Expected finite N1x192x192 restoration outputs.')\n    output = dmd_model.get_embedding(2*prediction[:,:,32:160,32:160]-1)\n    return output['feature'],output['mask']\n\ndef counterfactual_risks(before,after,threshold,temperature=.05,tail_fraction=.1):\n    if (before.ndim!=1 or before.shape!=after.shape or not len(before)\n            or not torch.isfinite(before).all() or not torch.isfinite(after).all()\n            or not math.isfinite(threshold) or temperature<=0 or not 0<tail_fraction<=1):\n        raise RuntimeError('Finite matched nonmated score vectors and valid risk constants required.')\n    increase = torch.relu(torch.sigmoid((after-threshold)/temperature)\n        -torch.sigmoid((before.detach()-threshold)/temperature))\n    count = max(1,math.ceil(tail_fraction*len(increase)))\n    return increase.mean(),increase.topk(count).values.mean()\n\ndef selected_query(record,bundle):\n    if bundle['status']!='matched':\n        raise RuntimeError('Unmatchable queries must be explicitly counted upstream.')\n    index = query_anchor_index(record['relative_path'],len(bundle['anchors']),0)\n    return index,bundle['feature'][index:index+1],bundle['mask'][index:index+1]\n"
exec(compile(LOCAL_SOURCE,'<dmd-local-scalar>','exec'))


In [ ]:
print('36 prerequisite: source34-and-warmstart',flush=True)

source34_config=json.loads((SOURCE34_DIR/'configuration.json').read_text(encoding='utf-8'))
source34_completion=json.loads((SOURCE34_DIR/'completion.json').read_text(encoding='utf-8'))
source34_sha=file_digest(SOURCE34_DIR/'configuration.json')
if (source34_completion['status']!='dmd_local_surrogate_complete'
        or SOURCE34_DIR.name!=source34_sha[:20]
        or source34_completion['configuration_sha256']!=source34_sha
        or source34_config['source33_configuration_sha256']!=source33_sha
        or source34_completion['permits_legacy_30_31'] is not False):
    raise RuntimeError('Expected unchanged completed34; no gate rewriting permitted.')
verify_files(SOURCE34_DIR,source34_completion['file_sha256'])
source34_completion_sha=file_digest(SOURCE34_DIR/'completion.json')
print('34 remains unchanged. Quality:',source34_completion['quality']['passed'])
SPECIALIZATION_ROOT=PROJECT_ROOT/'experiments'/'m5_dry_specialization_v1'
original_pointer=json.loads((SPECIALIZATION_ROOT/'completed_experiment.json').read_text(encoding='utf-8'))
ORIGINAL_DIR=Path(original_pointer['directory'])
if (ORIGINAL_DIR.parent!=SPECIALIZATION_ROOT
        or file_digest(ORIGINAL_DIR/'configuration.json')!=original_pointer['configuration_sha256']
        or file_digest(ORIGINAL_DIR/'completion.json')!=original_pointer['completion_sha256']):
    raise RuntimeError('Original23 pointer/configuration/completion changed.')
original_config=json.loads((ORIGINAL_DIR/'configuration.json').read_text(encoding='utf-8'))
original_completion=json.loads((ORIGINAL_DIR/'completion.json').read_text(encoding='utf-8'))
if (original_config['version']!='m5_dry_specialization_v1'
        or original_completion['status']!='dry_specialization_complete'
        or original_completion['configuration_sha256']!=original_pointer['configuration_sha256']
        or original_config['seed']!=SEED
        or original_config['initialization_seeds']!={'dry_l1_seed20261110':20261110}
        or original_config['selection_metric']!='same fixed validation mean global L1; initial identity checkpoint included'
        or original_config['losses']!={'dry_l1_seed20261110':'global L1'}
        or original_config['gradient_lambda']!=0
        or original_config['settings']!=SETTINGS):
    raise RuntimeError('Expected the original23 common seed10 L1 warmstart, without reselection.')
verify_files(ORIGINAL_DIR,original_completion['file_sha256'])
verify_files(ORIGINAL_DIR,original_config['source_file_sha256'])
for name,path in [('restoration_baseline.py',M0_DIR/'restoration_baseline.py'),
        ('train_sources.csv',M0_DIR/'train_sources.csv')]:
    if original_config['source_file_sha256'][name]!=file_digest(path):
        raise RuntimeError(f'Original23 source changed: {name}')
if original_config['simulator_configuration_sha256']!=file_digest(SIM_DIR/'configuration.json'):
    raise RuntimeError('Original23 simulator differs.')
for name,value in [('torch_version',str(torch.__version__)),('torch_cuda',str(torch.version.cuda)),
        ('numpy_version',np.__version__),('pillow_version',Image.__version__)]:
    if original_config[name]!=value:
        raise RuntimeError(f'Original23 runtime differs: {name}')
original_run=ORIGINAL_DIR/'dry_l1_seed20261110'
if file_digest(original_run/'completion.json')!=original_completion['run_completion_sha256']['dry_l1_seed20261110']:
    raise RuntimeError('Original23 selected run changed.')
original_run_completion=json.loads((original_run/'completion.json').read_text(encoding='utf-8'))
verify_files(original_run,original_run_completion['file_sha256'])
original_best=json.loads((original_run/'best.json').read_text(encoding='utf-8'))
if (original_best!=original_run_completion['result']['best_checkpoint']
        or Path(original_best['file']).name!=original_best['file']
        or original_best['configuration_sha256']!=original_pointer['configuration_sha256']
        or file_digest(original_run/original_best['file'])!=original_best['sha256']
        or original_run_completion['epochs_completed']!=8):
    raise RuntimeError('Original23 best checkpoint/budget differs.')
payload=torch.load(original_run/original_best['file'],map_location='cpu',weights_only=True)
if (payload['configuration_sha256']!=original_pointer['configuration_sha256']
        or payload['run']!='dry_l1_seed20261110' or payload['initialization_seed']!=20261110
        or not 0<=payload['completed_epochs']<=8):
    raise RuntimeError('Original23 safe payload differs.')
with torch.random.fork_rng(devices=[]):
    restorer=training.RestorationUNet()
restorer.load_state_dict(payload['model'],strict=True)
del payload
restorer.eval()
if sum(parameter.numel() for parameter in restorer.parameters())!=487009:
    raise RuntimeError('Original23 restoration architecture differs.')
if any(not torch.isfinite(value).all() for value in restorer.state_dict().values()):
    raise RuntimeError('Nonfinite original23 weights.')
initial_restorer_sha=state_sha(restorer)
print('Verified original23 selected warmstart. No optimizer or updates.')


In [ ]:

source35_sha=file_digest(SOURCE35_DIR/'configuration.json')
source35_config=json.loads((SOURCE35_DIR/'configuration.json').read_text(encoding='utf-8'))
source35_completion=json.loads((SOURCE35_DIR/'completion.json').read_text(encoding='utf-8'))
if (source35_sha!=EXPECTED35_SHA or SOURCE35_DIR.name!=source35_sha[:20]
        or source35_completion['status']!='frozen_before_lsar_check_complete'
        or source35_completion['configuration_sha256']!=source35_sha
        or source35_completion['permits_new_matched_pilot'] is not True
        or source35_completion['permits_legacy_30_31'] is not False
        or source35_config['source33_configuration_sha256']!=source33_sha
        or source35_config['source34_configuration_sha256']!=source34_sha
        or source35_config['source33_completion_sha256']!=file_digest(SOURCE33_DIR/'completion.json')
        or source35_config['source34_completion_sha256']!=source34_completion_sha
        or source35_config['original23_configuration_sha256']!=original_pointer['configuration_sha256']
        or source35_config['original23_completion_sha256']!=original_pointer['completion_sha256']
        or source35_config['original23_best']!=original_best):
    raise RuntimeError('Expected the exact passed35 and unchanged original23/33/34.')
verify_files(SOURCE35_DIR,source35_completion['file_sha256'])
source35_completion_sha=file_digest(SOURCE35_DIR/'completion.json')
frozen35_source=(SOURCE35_DIR/'frozen_lsar.py.txt').read_text(encoding='utf-8')
if hashlib.sha256(frozen35_source.encode()).hexdigest()!=source35_config['source_sha256']:
    raise RuntimeError('Frozen35 implementation hash changed.')
exec(compile(frozen35_source,'<verified35-functions>','exec'))
local_score,lsar_score=load_scoring(ASSET_ROOT/'evaluate_mnt.py')
source_thresholds=json.loads((SOURCE33_DIR/'thresholds.json').read_text(encoding='utf-8'))
source35_live=pd.read_csv(SOURCE35_DIR/'live_gradients.csv',keep_default_na=False)
galleries,calibration_probes=panel_records(partition,'calibration')
case_records=[]
for dataset,db in sorted({(p['dataset'],p['db']) for p in calibration_probes}):
    case_records.append(next(p for p in calibration_probes if (p['dataset'],p['db'])==(dataset,db)))
expected_probes=[p['relative_path'] for p in case_records]
if source35_live['probe'].tolist()!=expected_probes or not source35_live['passed'].eq(True).all():
    raise RuntimeError('Must use the exact same fixed successful35 calibration cases, not replacements.')
print('Verified passed35. Reusing its score and cases; no all-pair replay or full cache scan.',flush=True)


In [ ]:
IMAGE_SOURCE = "def checked_native_input(array):\n    if (array.ndim!=2 or min(array.shape)<8 or not np.isfinite(array).all()\n            or array.min()<0 or array.max()>1):\n        raise RuntimeError('Expected finite native-resolution grayscale image in [0,1].')\n    return torch.from_numpy(np.array(array,dtype=np.float32,copy=True))[None,None]\n\ndef check_native_output(output,image):\n    if (output.shape!=image.shape or not torch.isfinite(output).all()\n            or bool((output<0).any()) or bool((output>1).any())):\n        raise RuntimeError('Native restoration changed image shape/range/finiteness.')\n\n@torch.no_grad()\ndef native_prediction(array):\n    image=checked_native_input(array).to(CHECK_DEVICE)\n    output=restorer(image)\n    check_native_output(output,image)\n    return output.detach().cpu()\n\ndef descriptor_chunks(image,anchors):\n    if image.device.type!='cpu' or image.ndim!=4 or image.shape[:2]!=(1,1):\n        raise RuntimeError('Full-image DMD sampling and sampling backward must stay on CPU.')\n    if not 4<=len(anchors)<=24:\n        raise RuntimeError('Expected the same4..24 fixed original anchors.')\n    for start in range(0,len(anchors),CHUNK_SIZE):\n        patches=torch.cat([aligned_training_patch(image,a)[:,:,32:160,32:160]\n            for a in anchors[start:start+CHUNK_SIZE]])\n        values=dmd_model.get_embedding(2*patches.to(CHECK_DEVICE)-1)\n        yield start,values\n\n@torch.no_grad()\ndef native_descriptors(image,anchors):\n    features,masks=[],[]\n    for _,values in descriptor_chunks(image,anchors):\n        features.append(values['feature'].detach())\n        masks.append(values['mask'].detach())\n    return torch.cat(features),torch.cat(masks)\n\ndef image_vjps(image,anchors,feature_grads,mask_grads):\n    if not feature_grads or len(feature_grads)!=len(mask_grads):\n        raise RuntimeError('Paired nonempty descriptor cotangents required.')\n    leaf=image.detach().clone().requires_grad_()\n    sums=[torch.zeros_like(leaf) for _ in feature_grads]\n    for start,values in descriptor_chunks(leaf,anchors):\n        end=start+len(values['feature'])\n        for index,(df,dm) in enumerate(zip(feature_grads,mask_grads)):\n            grad,=torch.autograd.grad((values['feature'],values['mask']),leaf,\n                grad_outputs=(df[start:end],dm[start:end]),\n                retain_graph=index+1<len(feature_grads))\n            if not torch.isfinite(grad).all():\n                raise RuntimeError('Nonfinite DMD->native-image cotangent.')\n            sums[index].add_(grad.detach())\n    return sums\n\ndef native_parameter_vjps(degraded,reference,expected_output,image_grads):\n    image=checked_native_input(degraded).to(CHECK_DEVICE)\n    target=checked_native_input(reference).to(CHECK_DEVICE)\n    if image.shape!=target.shape or not image_grads:\n        raise RuntimeError('Native target/input shape mismatch or missing cotangents.')\n    output=restorer(image)\n    check_native_output(output,image)\n    recompute_error=float((output.detach().cpu()-expected_output).abs().max())\n    if recompute_error>1e-7:\n        raise RuntimeError('Native restoration recomputation differs from the scored image.')\n    parameters=tuple(restorer.parameters())\n    gradients=[]\n    for grad in image_grads:\n        if grad.shape!=output.shape:\n            raise RuntimeError('Full-image cotangent shape mismatch.')\n        gradients.append(tuple(g.detach() for g in torch.autograd.grad(output,parameters,\n            grad_outputs=grad.to(CHECK_DEVICE),retain_graph=True)))\n    loss=(output-target).abs().mean()\n    l1_grads=tuple(g.detach() for g in torch.autograd.grad(loss,parameters))\n    if not all(bool(torch.isfinite(g).all()) for group in gradients+[l1_grads] for g in group):\n        raise RuntimeError('Nonfinite native-image parameter gradients.')\n    return dict(objectives=gradients,l1_gradients=l1_grads,l1=float(loss.detach()),\n        recompute_error=recompute_error)\n\ndef gradient_norms(gradients):\n    return dict(l1=sum(float(g.abs().sum()) for g in gradients),\n        l2=math.sqrt(sum(float(g.square().sum()) for g in gradients)))\n\ndef native_quantize(image):\n    array=image.detach().cpu()[0,0].numpy()\n    if not np.isfinite(array).all() or array.min()<0 or array.max()>1:\n        raise RuntimeError('Cannot quantize invalid native image.')\n    return np.rint(array*255).astype(np.uint8)\n"
exec(compile(IMAGE_SOURCE,'<matched-native-image>','exec'))


In [ ]:

SOURCE36_DIR=PROJECT_ROOT/'diagnostics'/'matched_native_image_check_v1'/'538b2d164a98d82c81b8'
EXPECTED36_SHA='538b2d164a98d82c81b8c178731d13bab20475cc02f1f86c1c3dbcc4e9bb11de'
source36_config=json.loads((SOURCE36_DIR/'configuration.json').read_text(encoding='utf-8'))
source36_completion=json.loads((SOURCE36_DIR/'completion.json').read_text(encoding='utf-8'))
if (file_digest(SOURCE36_DIR/'configuration.json')!=EXPECTED36_SHA
        or source36_completion['configuration_sha256']!=EXPECTED36_SHA
        or source36_completion['status']!='matched_native_image_check_complete'
        or source36_completion['permits_preparing_native_image_pilot'] is not True
        or source36_completion['training_performed'] is not False
        or source36_config['source35_configuration_sha256']!=source35_sha
        or source36_config['source35_completion_sha256']!=source35_completion_sha
        or source36_config['original23_configuration_sha256']!=original_pointer['configuration_sha256']
        or source36_config['original23_best']!=original_best
        or source36_config['image_source_sha256']!=hashlib.sha256(IMAGE_SOURCE.encode()).hexdigest()):
    raise RuntimeError('Need exact passed36 and its unchanged native-image implementation.')
verify_files(SOURCE36_DIR,source36_completion['file_sha256'])
source36_completion_sha=file_digest(SOURCE36_DIR/'completion.json')
warm_model_state={key:value.detach().cpu().clone() for key,value in restorer.state_dict().items()}
print('Passed36 verified. ACTUAL training is authorized only for this new matched pilot.')


In [ ]:
WORK_DIR=LOCAL_ROOT/'native_pilot_nbis'
REPORT_DIR=WORK_DIR/'reports'
WORK_DIR.mkdir(parents=True,exist_ok=True)
REPORT_DIR.mkdir(parents=True,exist_ok=True)
NBIS_URL = 'https://nigos.nist.gov/nist/nbis/nbis_v5_0_0.zip'
NBIS_SHA256 = '0adf8ab0f6b0e4208de50ca00ba21d3d77112ecd66288757ddfed21f6bee92c3'
NBIS_BYTES = 52595795
ARCHIVE_DIR = PROJECT_ROOT / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
NBIS_ARCHIVE = ARCHIVE_DIR / 'nbis_v5_0_0.zip'
missing_tools = [name for name in ('curl', 'gcc', 'make', 'sh') if shutil.which(name) is None]
if missing_tools:
    raise RuntimeError(f'Missing build tools: {missing_tools}. Install required tools before retrying.')
if not NBIS_ARCHIVE.exists():
    partial = LOCAL_ROOT / 'nbis_v5_0_0.zip.part'
    if not partial.exists() or partial.stat().st_size != NBIS_BYTES:
        subprocess.run([
            'curl', '--fail', '--location', '--continue-at', '-',
            '--connect-timeout', '30', '--max-time', '600',
            '--output', str(partial), NBIS_URL,
        ], check=True)
    if partial.stat().st_size != NBIS_BYTES or file_digest(partial) != NBIS_SHA256:
        raise RuntimeError('NBIS download integrity failure. Partial file retained; do not build it.')
    pending = NBIS_ARCHIVE.with_suffix('.zip.part')
    shutil.copyfile(partial, pending)
    if file_digest(pending) != NBIS_SHA256:
        raise RuntimeError('NBIS Drive copy checksum mismatch.')
    pending.replace(NBIS_ARCHIVE)
    partial.unlink()
if NBIS_ARCHIVE.stat().st_size != NBIS_BYTES or file_digest(NBIS_ARCHIVE) != NBIS_SHA256:
    raise RuntimeError('Cached NBIS archive differs from the verified release.')
print('Official NBIS source verified:', NBIS_SHA256)
BUILD_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
SOURCE_ROOT = BUILD_ROOT / 'source'
PREFIX = BUILD_ROOT / 'install'
SOURCE = SOURCE_ROOT / 'Rel_5.0.0'
MINDTCT = PREFIX / 'bin' / 'mindtct'
BOZORTH3 = PREFIX / 'bin' / 'bozorth3'
STAMP = BUILD_ROOT / 'verified_build.json'
build_commands = [
    ['sh', 'setup.sh', str(PREFIX), '--without-X11', '--without-OPENJP2', '--64'],
    ['make', 'config'],
    ['make', 'it', 'CC=gcc -fcommon'],
    ['make', 'install', 'LIBNBIS=no', 'CC=gcc -fcommon'],
]
build_signature = {'source_sha256': NBIS_SHA256, 'commands': build_commands}
if STAMP.exists():
    build_record = json.loads(STAMP.read_text(encoding='utf-8'))
    if build_record['signature'] != build_signature:
        raise RuntimeError('Cached NBIS build configuration differs; do not reuse it.')
    for binary in (MINDTCT, BOZORTH3):
        if not binary.is_file() or file_digest(binary) != build_record['binary_sha256'][binary.name]:
            raise RuntimeError(f'Cached NBIS binary changed or is missing: {binary}')
    print('Reusing verified local NBIS build.')
else:
    SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
    PREFIX.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(NBIS_ARCHIVE) as archive:
        for member in archive.infolist():
            path = PurePosixPath(member.filename)
            if path.is_absolute() or '..' in path.parts or '\\' in member.filename:
                raise RuntimeError(f'Unsafe source archive path: {member.filename}')
        archive.extractall(SOURCE_ROOT)
    for path in SOURCE.rglob('*'):
        if path.is_file() and (path.name == 'configure' or path.suffix == '.sh'):
            path.chmod(path.stat().st_mode | 0o111)
    build_log = WORK_DIR / 'build.log'
    with build_log.open('w', encoding='utf-8') as output:
        for command in build_commands:
            print('Building:', ' '.join(command), flush=True)
            result = subprocess.run(command, cwd=SOURCE, stdout=output, stderr=subprocess.STDOUT)
            output.flush()
            if result.returncode:
                shutil.copyfile(build_log, REPORT_DIR / 'build.log')
                print('\n'.join(build_log.read_text(errors='replace').splitlines()[-60:]))
                raise RuntimeError(f'NBIS build failed with code {result.returncode}. See saved build.log.')
    shutil.copyfile(build_log, REPORT_DIR / 'build.log')
    build_record = {
        'signature': build_signature,
        'binary_sha256': {binary.name: file_digest(binary) for binary in (MINDTCT, BOZORTH3)},
        'compiler': subprocess.run(['gcc', '--version'], check=True, capture_output=True, text=True).stdout,
        'platform': platform.platform(),
    }
    STAMP.write_text(json.dumps(build_record, indent=2), encoding='utf-8')
versions = {}
for binary in (MINDTCT, BOZORTH3):
    result = subprocess.run([str(binary), '-version'], check=True, capture_output=True, text=True)
    versions[binary.name] = result.stdout + result.stderr
    print(binary.name, versions[binary.name].strip())
(REPORT_DIR / 'build_provenance.json').write_text(
    json.dumps({**build_record, 'versions': versions}, indent=2), encoding='utf-8')
if build_record['binary_sha256']!=source_config['nbis_build']['binary_sha256']:
    raise RuntimeError('NBIS binaries differ from frozen33 anchors. No mixing or threshold fallback.')


In [ ]:
BANK_SOURCE = 'def safe_pilot_name(name):\n    if not isinstance(name,str) or not name or Path(name).name!=name or \'/\' in name or \'\\\\\' in name or \':\' in name or name in (\'.\',\'..\'):\n        raise RuntimeError(\'Unsafe pilot artifact filename.\')\n    return name\n\ndef atomic_torch(path,payload):\n    local=LOCAL_PILOT/f\'{path.name}.part\'\n    torch.save(payload,local)\n    digest=file_digest(local)\n    pending=path.with_suffix(path.suffix+\'.part\')\n    shutil.copyfile(local,pending)\n    if file_digest(pending)!=digest:\n        raise RuntimeError(f\'Drive tensor copy checksum failed: {path}\')\n    pending.replace(path)\n    local.unlink()\n    return digest\n\ndef load_safe_tensor(path,sha):\n    if file_digest(path)!=sha:\n        raise RuntimeError(f\'Packed bank/checkpoint checksum differs: {path}\')\n    return torch.load(path,map_location=\'cpu\',weights_only=True)\n\ndef bank_entry(record):\n    relative=record[\'relative_path\']\n    if relative not in allowed_train_paths:\n        raise RuntimeError(\'Pilot bank may only use main TRAIN.\')\n    shared=SOURCE33_DIR/\'anchors\'/hashlib.sha256(relative.encode()).hexdigest()[:24]\n    if (shared/\'completion.json\').exists():\n        expected=dict(relative_path=relative,source_sha256=record[\'sha256\'],\n            configuration_sha256=source33_sha,mindtct_sha256=build_record[\'binary_sha256\'][\'mindtct\'])\n        cached_bundle(shared,expected)\n        with np.load(shared/\'anchors.npz\',allow_pickle=False) as archive:\n            anchors=archive[\'anchors\'].copy()\n            shape=archive[\'image_shape\'].copy()\n        if not np.array_equal(anchors,parse_nbis_xyt((shared/\'minutiae.xyt\').read_text(encoding=\'ascii\'),*shape)):\n            raise RuntimeError(\'Original33 anchors disagree with their XYT source.\')\n        reference=original_train_image(record)\n    else:\n        reference=original_train_image(record)\n        root=LOCAL_PILOT/\'anchor\'\n        image_path=root.with_suffix(\'.png\')\n        Image.fromarray(np.rint(reference*255).astype(np.uint8)).save(image_path,dpi=(500,500))\n        if root.with_suffix(\'.xyt\').exists():\n            root.with_suffix(\'.xyt\').unlink()\n        result=subprocess.run([str(MINDTCT),str(image_path),str(root)],\n            capture_output=True,text=True,timeout=120)\n        if result.returncode or not root.with_suffix(\'.xyt\').is_file():\n            log=LOCAL_PILOT/\'mindtct_error.log\'\n            log.write_text(result.stdout+\'\\n\'+result.stderr,encoding=\'utf-8\')\n            raise RuntimeError(f\'MINDTCT failed for {relative}; see {log}\')\n        text=root.with_suffix(\'.xyt\').read_text(encoding=\'ascii\')\n        anchors=parse_nbis_xyt(text,*reference.shape)\n        shape=np.array(reference.shape)\n    entry=dict(relative_path=relative,source_sha256=record[\'sha256\'],\n        anchors=torch.from_numpy(anchors.copy()),image_shape=torch.as_tensor(shape.copy()),\n        status=\'matched\' if len(anchors)>=4 else \'unmatchable_original_anchor_count\')\n    if len(anchors)>=4:\n        feature,mask=native_descriptors(checked_native_input(reference),anchors)\n        entry.update(feature=feature.cpu(),mask=mask.cpu())\n    return entry\n\ndef validate_bank_entry(entry,record):\n    if (entry[\'relative_path\']!=record[\'relative_path\'] or entry[\'source_sha256\']!=record[\'sha256\']\n            or entry[\'anchors\'].ndim!=2 or entry[\'anchors\'].shape[1]!=4\n            or len(entry[\'anchors\'])>24 or not torch.isfinite(entry[\'anchors\']).all()\n            or entry[\'status\']!=(\'matched\' if len(entry[\'anchors\'])>=4 else \'unmatchable_original_anchor_count\')):\n        raise RuntimeError(\'Packed TRAIN bank record/anchor provenance differs.\')\n    if entry[\'status\']==\'matched\':\n        feature,mask=entry[\'feature\'],entry[\'mask\']\n        if (feature.shape!=(len(entry[\'anchors\']),768) or mask.shape!=(len(entry[\'anchors\']),64)\n                or not torch.isfinite(feature).all() or not torch.isfinite(mask).all()\n                or not ((mask>=0)&(mask<=1)).all()):\n            raise RuntimeError(\'Invalid packed TRAIN DMD features.\')\n\ndef prepare_bank(records):\n    bank={}\n    shards={}\n    for offset in range(0,len(records),64):\n        selected=records[offset:offset+64]\n        name=f\'bank_{offset:05d}.pt\'\n        marker=BANK_DIR/f\'{name}.json\'\n        if marker.exists():\n            saved=json.loads(marker.read_text(encoding=\'utf-8\'))\n            if saved[\'configuration_sha256\']!=bank_sha or saved[\'file\']!=name:\n                raise RuntimeError(\'Bank shard marker differs.\')\n            entries=load_safe_tensor(BANK_DIR/name,saved[\'sha256\'])\n        else:\n            entries=[]\n            for index,record in enumerate(selected,1):\n                entries.append(bank_entry(record))\n                if index%16==0:\n                    print(f\'Building TRAIN bank shard: {offset+index}/{len(records)}\',flush=True)\n            digest=atomic_torch(BANK_DIR/name,entries)\n            saved=dict(configuration_sha256=bank_sha,file=name,sha256=digest)\n            write_verified(marker,json_bytes(saved))\n        if len(entries)!=len(selected):\n            raise RuntimeError(\'Bank shard coverage differs.\')\n        for entry,record in zip(entries,selected):\n            validate_bank_entry(entry,record)\n            bank[record[\'relative_path\']]=entry\n        shards[name]=saved[\'sha256\']\n        print(f\'TRAIN anchor/descriptor bank: {min(offset+64,len(records))}/{len(records)} verified\',flush=True)\n    return bank,shards\n\ndef fixed_targets(record):\n    own=bank37[record[\'relative_path\']]\n    same=sorted((r for r in bank_records if r[\'identity_key\']==record[\'identity_key\']\n        and r[\'sha256\']!=record[\'sha256\'] and bank37[r[\'relative_path\']][\'status\']==\'matched\'),\n        key=lambda r:r[\'relative_path\'])\n    candidates={}\n    for row in gallery37:\n        if ((row[\'dataset\'],row[\'db\'])==(record[\'dataset\'],record[\'db\'])\n                and row[\'identity_key\']!=record[\'identity_key\']\n                and groups37[row[\'identity_key\']]!=groups37[record[\'identity_key\']]):\n            candidates.setdefault(groups37[row[\'identity_key\']],row)\n    negatives=sorted(candidates.values(),key=lambda row:training.stable_seed(\n        f"{SEED}|native37_negative|{record[\'relative_path\']}|{row[\'relative_path\']}"))[:7]\n    if own[\'status\']!=\'matched\' or not same or len(negatives)!=7:\n        return None\n    return same[0],negatives\n'
exec(compile(BANK_SOURCE,'<native-pilot-bank>','exec'))


In [ ]:

PILOT_ROOT=PROJECT_ROOT/'experiments'/'native_triplet_replication_v1'
BANK_ROOT=PROJECT_ROOT/'experiments'/'native_counterfactual_pilot_v1'/'banks'
LOCAL_PILOT=LOCAL_ROOT/'native_counterfactual_pilot'
LOCAL_PILOT.mkdir(parents=True,exist_ok=True)
bank_config=dict(version='native_train_dmd_bank_v1',train_sources_sha256=file_digest(M0_DIR/'train_sources.csv'),
    source33_configuration_sha256=source33_sha,source36_configuration_sha256=EXPECTED36_SHA,
    nbis_binary_sha256=build_record['binary_sha256'],source_sha256=hashlib.sha256(BANK_SOURCE.encode()).hexdigest(),
    image_source_sha256=hashlib.sha256(IMAGE_SOURCE.encode()).hexdigest(),
    max_anchors=24,minimum_anchors=4,shard_size=64)
bank_sha=hashlib.sha256(json_bytes(bank_config)).hexdigest()
BANK_DIR=BANK_ROOT/bank_sha[:20]
BANK_DIR.mkdir(parents=True,exist_ok=True)
if (BANK_DIR/'configuration.json').exists() and (BANK_DIR/'configuration.json').read_bytes()!=json_bytes(bank_config):
    raise RuntimeError('Packed bank configuration changed.')
write_verified(BANK_DIR/'configuration.json',json_bytes(bank_config))
dmd_model.to(CHECK_DEVICE)
bank_records=train_sources.sort_values('relative_path').to_dict('records')
bank37,bank_shards=prepare_bank(bank_records)
groups37=matcher.identity_groups(train_sources)
gallery37=[record for _,frame in train_sources.groupby('identity_key',sort=True)
    for record in [next((r for r in frame.sort_values('relative_path').to_dict('records')
        if bank37[r['relative_path']]['status']=='matched'),None)] if record is not None]
target_map={record['relative_path']:fixed_targets(record) for record in bank_records}
eligibility=pd.DataFrame([dict(relative_path=r['relative_path'],status=bank37[r['relative_path']]['status'],
    auxiliary_available=target_map[r['relative_path']] is not None) for r in bank_records])
write_csv(BANK_DIR/'eligibility.csv',eligibility)
bank_completion=dict(configuration_sha256=bank_sha,
    file_sha256={**bank_shards,'configuration.json':file_digest(BANK_DIR/'configuration.json'),
        'eligibility.csv':file_digest(BANK_DIR/'eligibility.csv')})
write_verified(BANK_DIR/'completion.json',json_bytes(bank_completion))
print('TRAIN bank status (unavailable auxiliary is EXPLICIT L1-only, no task drops):')
print(eligibility.groupby(['status','auxiliary_available']).size().to_string())


In [ ]:

for name in ('task_condition_audit.csv','order_shared.csv'):
    if file_digest(ORIGINAL_DIR/name)!=original_config['source_file_sha256'][name]:
        raise RuntimeError(f'Original23 task schedule changed: {name}')
audit37=pd.read_csv(ORIGINAL_DIR/'task_condition_audit.csv',keep_default_na=False).set_index('task_id')
order37=pd.read_csv(ORIGINAL_DIR/'order_shared.csv')['task_id'].tolist()
if (len(order37)!=len(audit37) or len(set(order37))!=len(order37)
        or set(order37)!=set(audit37.index) or len(order37)!=8*SOURCE_COUNT):
    raise RuntimeError('Expected the complete original23 task permutation.')
selected_ids=order37[:PILOT_EPOCHS*SOURCE_COUNT]
schedule=audit37.loc[selected_ids].reset_index()
if not set(schedule['relative_path']).issubset(allowed_train_paths):
    raise RuntimeError('Pilot schedule must stay inside main TRAIN.')
if not set(schedule['specialized_condition']).issubset(
        {'identity/none','dry_like/mild','dry_like/moderate','dry_like/severe'}):
    raise RuntimeError('Expected original23 dry-specialized task conditions.')
source_by_path={r['relative_path']:r for r in bank_records}
for row in schedule.to_dict('records'):
    row_source=source_by_path[row['relative_path']]
    expected_source=train_sources.iloc[int(row['task_id'])%SOURCE_COUNT]['relative_path']
    if row_source['relative_path']!=expected_source:
        raise RuntimeError('Original23 task/source ordering differs.')
validation_dataset=training.FingerprintPairs(validation_sources.to_dict('records'),DATA_ROOT,simulator,
    pair_config['source_dpi'],SEED,patch_size=SETTINGS['patch_size'],training=False)
validation_loader=DataLoader(validation_dataset,batch_size=SETTINGS['batch_size'],shuffle=False,
    num_workers=0,generator=torch.Generator().manual_seed(SEED))

SELECTOR_SOURCE = "@torch.inference_mode()\ndef evaluate(model):\n    model.eval()\n    totals = {}\n    for batch in validation_loader:\n        xs, ys = (batch['input'].to(DEVICE), batch['target'].to(DEVICE))\n        predictions = model(xs).float()\n        if predictions.shape != ys.shape or not torch.isfinite(predictions).all() or predictions.min() < 0 or (predictions.max() > 1):\n            raise RuntimeError('Validation output shape/range/finiteness failure.')\n        png_before, png_after, png_target = (torch.round(xs * 255), torch.round(predictions * 255), torch.round(ys * 255))\n        columns = {'mae': (predictions - ys).abs().mean((1, 2, 3)), 'mse': (predictions - ys).square().mean((1, 2, 3)), 'input_mae': (xs - ys).abs().mean((1, 2, 3)), 'png_mae': (png_after - png_target).abs().mean((1, 2, 3)) / 255, 'png_changed_fraction': png_after.ne(png_before).float().mean((1, 2, 3)), 'png_identical_image_fraction': png_after.eq(png_before).flatten(1).all(1).float()}\n        values = {name: tensor.cpu().tolist() for name, tensor in columns.items()}\n        for index, condition in enumerate(batch['condition']):\n            total = totals.setdefault(condition, {'samples': 0, **{name: 0.0 for name in columns}})\n            total['samples'] += 1\n            for name in columns:\n                total[name] += values[name][index]\n    expected = {f'{family}/{severity}' for family, severity in all_conditions}\n    if set(totals) != expected or any((row['samples'] != len(validation_sources) for row in totals.values())):\n        raise RuntimeError('Validation must cover every frozen source in all ten conditions.')\n    rows = [{'condition': condition, 'samples': total['samples'], **{name: value / total['samples'] for name, value in total.items() if name != 'samples'}} for condition, total in sorted(totals.items())]\n    count = sum((row['samples'] for row in rows))\n    result = {name: sum((row[name] * row['samples'] for row in rows)) / count for name in ('mae', 'mse', 'input_mae', 'png_mae', 'png_changed_fraction', 'png_identical_image_fraction')}\n    result.update(identity_mae=next((row['mae'] for row in rows if row['condition'] == 'identity/none')), conditions=rows)\n    return result"
exec(compile(SELECTOR_SOURCE,'<unchanged-global-L1-selector>','exec'))


In [ ]:
TRAIN_SOURCE = 'def task_input(task):\n    record=source_by_path[task[\'relative_path\']]\n    reference=original_train_image(record)\n    family,severity=task[\'specialized_condition\'].split(\'/\')\n    if family==\'identity\':\n        return record,reference,reference.copy()\n    seed=simulator.case_seed(record[\'relative_path\'],family,severity,\n        training.stable_seed(f"{REPLICATION_SEED}|native37|{int(task[\'source_epoch\'])}"))\n    degraded,_=simulator.degrade(reference,family,severity,seed)\n    return record,reference,degraded\n\ndef native_task_gradient(task,arm):\n    if arm not in PILOT_RUNS:\n        raise RuntimeError(\'Unknown native-pilot objective.\')\n    record,reference,degraded=task_input(task)\n    image=native_prediction(degraded)\n    selected=target_map[record[\'relative_path\']]\n    auxiliary_values=dict(triplet=0.0,mean_risk=0.0,tail_risk=0.0)\n    if selected is None:\n        image_grad=torch.zeros_like(image)\n        status=\'auxiliary_unavailable\'\n    else:\n        positive,negatives=selected\n        query=bank37[record[\'relative_path\']]\n        anchors=query[\'anchors\'].numpy()\n        bf,bm=native_descriptors(checked_native_input(degraded),anchors)\n        af,am=native_descriptors(image,anchors)\n        feature=af.detach().requires_grad_()\n        mask=am.detach().requires_grad_()\n        targets=[bank37[r[\'relative_path\']] for r in [positive]+negatives]\n        plans=[freeze_lsar(local_matrix(bf.cpu(),bm.cpu(),target),query[\'anchors\'],target[\'anchors\'])\n            for target in targets]\n        after=torch.stack([frozen_lsar_score(plan,local_matrix(feature,mask,target))\n            for plan,target in zip(plans,targets)])\n        before=torch.stack([plan[\'baseline\'] for plan in plans]).to(CHECK_DEVICE)\n        threshold=source_thresholds[f"{record[\'dataset\']}_{record[\'db\']}"]\n        mean,tail=counterfactual_risks(before[1:],after[1:],threshold)\n        triplet=(.2+after[1:].max()-after[0]).clamp_min(0)\n        auxiliary_values=dict(triplet=float(triplet.detach()),mean_risk=float(mean.detach()),\n            tail_risk=float(tail.detach()))\n        objective={\'native_l1\':after.mean()*0,\'native_triplet\':triplet,\n            \'native_mean\':mean,\'native_tail\':tail}[arm]\n        df,dm=torch.autograd.grad(objective,(feature,mask))\n        image_grad=image_vjps(image,anchors,[df.detach()],[dm.detach()])[0]\n        status=\'auxiliary_available\'\n    chain=native_parameter_vjps(degraded,reference,image,[image_grad])\n    gradients=tuple(l1+.1*aux for l1,aux in zip(chain[\'l1_gradients\'],chain[\'objectives\'][0]))\n    if not all(bool(torch.isfinite(grad).all()) for grad in gradients):\n        raise RuntimeError(\'Nonfinite combined native-pilot gradient.\')\n    auxiliary={\'native_l1\':0.0,\'native_triplet\':auxiliary_values[\'triplet\'],\n        \'native_mean\':auxiliary_values[\'mean_risk\'],\'native_tail\':auxiliary_values[\'tail_risk\']}[arm]\n    metrics=dict(l1=chain[\'l1\'],total=chain[\'l1\']+.1*auxiliary,**auxiliary_values)\n    return gradients,metrics,status\n\ndef snapshot_model(model):\n    return {key:value.detach().cpu().clone() for key,value in model.state_dict().items()}\n\ndef save_native_state(directory,state):\n    digest=atomic_torch(directory/\'pending.pt\',state)\n    filename=f"step_{state[\'updates\']:05d}_{digest[:20]}.pt"\n    source=directory/\'pending.pt\'\n    destination=directory/filename\n    if destination.exists() and file_digest(destination)!=digest:\n        raise RuntimeError(\'Immutable native checkpoint collision.\')\n    source.replace(destination)\n    write_verified(directory/\'latest.json\',json_bytes(dict(file=filename,sha256=digest,\n        configuration_sha256=experiment_sha,run=directory.name,updates=state[\'updates\'])))\n\ndef load_native_state(directory):\n    path=directory/\'latest.json\'\n    if not path.exists():\n        return None\n    pointer=json.loads(path.read_text(encoding=\'utf-8\'))\n    name=safe_pilot_name(pointer[\'file\'])\n    if pointer[\'configuration_sha256\']!=experiment_sha or pointer[\'run\']!=directory.name:\n        raise RuntimeError(\'Native resume pointer differs.\')\n    state=load_safe_tensor(directory/name,pointer[\'sha256\'])\n    if (state[\'configuration_sha256\']!=experiment_sha or state[\'run\']!=directory.name\n            or state[\'updates\']!=pointer[\'updates\'] or not 0<=state[\'epoch\']<=PILOT_EPOCHS\n            or not 0<=state[\'offset\']<SOURCE_COUNT or len(state[\'history\'])!=state[\'epoch\']\n            or (state[\'epoch\']==PILOT_EPOCHS and state[\'offset\']!=0)):\n        raise RuntimeError(\'Native resume state/budget differs.\')\n    expected_updates=state[\'epoch\']*math.ceil(SOURCE_COUNT/EFFECTIVE_BATCH)+math.ceil(state[\'offset\']/EFFECTIVE_BATCH)\n    if state[\'updates\']!=expected_updates or state[\'seen\']!=state[\'offset\']:\n        raise RuntimeError(\'Native resume task offset/optimizer dose differs.\')\n    return state\n\ndef train_native_arm(arm):\n    global restorer\n    if arm not in PILOT_RUNS:\n        raise RuntimeError(\'Unknown native pilot arm.\')\n    directory=EXPERIMENT_DIR/arm\n    directory.mkdir(parents=True,exist_ok=True)\n    if (directory/\'completion.json\').exists():\n        completion_arm=json.loads((directory/\'completion.json\').read_text(encoding=\'utf-8\'))\n        if (completion_arm[\'configuration_sha256\']!=experiment_sha or completion_arm[\'run\']!=arm\n                or completion_arm[\'epochs_completed\']!=PILOT_EPOCHS):\n            raise RuntimeError(\'Completed native arm configuration/budget differs.\')\n        verify_files(directory,completion_arm[\'file_sha256\'])\n        print(\'Reused completed native pilot arm:\',arm,flush=True)\n        return completion_arm\n    with torch.random.fork_rng(devices=[]):\n        restorer=training.RestorationUNet()\n    restorer.load_state_dict(warm_model_state,strict=True)\n    restorer.to(CHECK_DEVICE)\n    optimizer=torch.optim.Adam(restorer.parameters(),lr=SETTINGS[\'learning_rate\'])\n    state=load_native_state(directory)\n    if state is None:\n        initial=evaluate(restorer)\n        if abs(initial[\'mae\']-original_best[\'validation_mae\'])>1e-7:\n            raise RuntimeError(\'Common original23 warmstart selector replay differs.\')\n        state=dict(configuration_sha256=experiment_sha,run=arm,epoch=0,offset=0,updates=0,\n            seen=0,sums={key:0.0 for key in (\'l1\',\'total\',\'triplet\',\'mean_risk\',\'tail_risk\')},\n            unavailable=0,history=[],best_epoch=0,best_mae=initial[\'mae\'],\n            best_model=snapshot_model(restorer),initial_validation=initial)\n    else:\n        restorer.load_state_dict(state[\'model\'],strict=True)\n        optimizer.load_state_dict(state[\'optimizer\'])\n        torch.set_rng_state(state[\'torch_rng\'])\n        torch.cuda.set_rng_state_all(state[\'cuda_rng\'])\n    def commit():\n        state.update(model=snapshot_model(restorer),optimizer=optimizer.state_dict(),\n            torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all())\n        save_native_state(directory,state)\n    if not (directory/\'latest.json\').exists():\n        commit()\n    arm_started=time.monotonic()\n    resumed_updates=state[\'updates\']\n    while state[\'epoch\']<PILOT_EPOCHS:\n        epoch=state[\'epoch\']\n        rows=schedule.iloc[epoch*SOURCE_COUNT:(epoch+1)*SOURCE_COUNT].to_dict(\'records\')\n        if CHECK_DEVICE.type==\'cuda\':\n            torch.cuda.reset_peak_memory_stats()\n        while state[\'offset\']<SOURCE_COUNT:\n            batch=rows[state[\'offset\']:state[\'offset\']+EFFECTIVE_BATCH]\n            optimizer.zero_grad(set_to_none=True)\n            restorer.eval()\n            for task in batch:\n                gradients,metrics,status=native_task_gradient(task,arm)\n                for parameter,gradient in zip(restorer.parameters(),gradients):\n                    if parameter.grad is None:\n                        parameter.grad=gradient/len(batch)\n                    else:\n                        parameter.grad.add_(gradient,alpha=1/len(batch))\n                for key,value in metrics.items():\n                    if not math.isfinite(value):\n                        raise RuntimeError(\'Nonfinite native task metric.\')\n                    state[\'sums\'][key]+=value\n                state[\'unavailable\']+=int(status==\'auxiliary_unavailable\')\n            optimizer.step()\n            if any(not torch.isfinite(value).all() for value in restorer.state_dict().values()):\n                raise RuntimeError(\'Native optimizer produced nonfinite parameters.\')\n            state[\'offset\']+=len(batch)\n            state[\'seen\']+=len(batch)\n            state[\'updates\']+=1\n            if state[\'updates\']%CHECKPOINT_EVERY==0 and state[\'offset\']<SOURCE_COUNT:\n                commit()\n                elapsed=time.monotonic()-arm_started\n                completed=state[\'updates\']-resumed_updates\n                remaining=PILOT_EPOCHS*math.ceil(SOURCE_COUNT/EFFECTIVE_BATCH)-state[\'updates\']\n                print(f"{arm}: epoch{epoch+1}/{PILOT_EPOCHS}, images{state[\'offset\']}/{SOURCE_COUNT}, "\n                    f"updates{state[\'updates\']}, L1={state[\'sums\'][\'l1\']/state[\'seen\']:.6f}, "\n                    f"aux_unavailable={state[\'unavailable\']}, "\n                    f"elapsed={elapsed/60:.1f}min, training_ETA~{elapsed/max(1,completed)*remaining/60:.1f}min, "\n                    f"peak_allocated_MiB={torch.cuda.max_memory_allocated()/2**20 if CHECK_DEVICE.type==\'cuda\' else 0:.1f}",flush=True)\n        validation=evaluate(restorer)\n        if validation[\'mae\']<state[\'best_mae\']:\n            state.update(best_mae=validation[\'mae\'],best_epoch=epoch+1,best_model=snapshot_model(restorer))\n        state[\'history\'].append(dict(epoch=epoch+1,**{f\'train_{key}\':value/state[\'seen\']\n            for key,value in state[\'sums\'].items()},auxiliary_unavailable=state[\'unavailable\'],\n            validation_mae=validation[\'mae\'],best_epoch=state[\'best_epoch\'],\n            peak_allocated_bytes=torch.cuda.max_memory_allocated() if CHECK_DEVICE.type==\'cuda\' else 0,\n            peak_reserved_bytes=torch.cuda.max_memory_reserved() if CHECK_DEVICE.type==\'cuda\' else 0))\n        state.update(epoch=epoch+1,offset=0,seen=0,unavailable=0,sums={key:0.0 for key in state[\'sums\']})\n        commit()\n        write_csv(directory/\'history.csv\',pd.DataFrame(state[\'history\']))\n        print(f"{arm}: saved epoch{state[\'epoch\']}, selected={state[\'best_epoch\']}, "\n            f"validation_L1={validation[\'mae\']:.6f}",flush=True)\n    restorer.load_state_dict(state[\'best_model\'],strict=True)\n    selected_validation=evaluate(restorer)\n    if abs(selected_validation[\'mae\']-state[\'best_mae\'])>1e-7:\n        raise RuntimeError(\'Selected native checkpoint validation replay differs.\')\n    selected=dict(configuration_sha256=experiment_sha,run=arm,completed_epochs=state[\'best_epoch\'],\n        trained_epochs=PILOT_EPOCHS,model=snapshot_model(restorer))\n    sha=atomic_torch(directory/\'selected.pt\',selected)\n    pointer=dict(file=\'selected.pt\',sha256=sha,configuration_sha256=experiment_sha,\n        completed_epochs=state[\'best_epoch\'],validation_mae=state[\'best_mae\'])\n    write_verified(directory/\'best.json\',json_bytes(pointer))\n    write_verified(directory/\'best_validation.json\',json_bytes(selected_validation))\n    write_csv(directory/\'history.csv\',pd.DataFrame(state[\'history\']))\n    marker=dict(configuration_sha256=experiment_sha,run=arm,epochs_completed=PILOT_EPOCHS,\n        best_checkpoint=pointer,file_sha256={name:file_digest(directory/name)\n            for name in (\'selected.pt\',\'best.json\',\'best_validation.json\',\'history.csv\',\'latest.json\')})\n    write_verified(directory/\'completion.json\',json_bytes(marker))\n    return marker\n'
exec(compile(TRAIN_SOURCE,'<native-pilot-training>','exec'))


In [ ]:

experiment_config=dict(version='native_triplet_replication_v1',implementation_sha256='e59c96842f759c3dcbf563ab44660611e3225688801cb0eab3de3a50f9755232',
    run_names=list(PILOT_RUNS),epochs=PILOT_EPOCHS,effective_batch=EFFECTIVE_BATCH,seed=REPLICATION_SEED,
    frozen_selector_seed=SEED,
    learning_rate=SETTINGS['learning_rate'],source_count=SOURCE_COUNT,
    updates_per_run=PILOT_EPOCHS*math.ceil(SOURCE_COUNT/EFFECTIVE_BATCH),
    source36_configuration_sha256=EXPECTED36_SHA,source36_completion_sha256=source36_completion_sha,
    source35_configuration_sha256=source35_sha,bank_configuration_sha256=bank_sha,
    bank_completion_sha256=file_digest(BANK_DIR/'completion.json'),
    original23_configuration_sha256=original_pointer['configuration_sha256'],
    original23_best=original_best,schedule_sha256=hashlib.sha256(schedule.to_csv(index=False,lineterminator='\n').encode()).hexdigest(),
    train_sources_sha256=file_digest(M0_DIR/'train_sources.csv'),
    validation_sources_sha256=file_digest(M0_DIR/'validation_sources.csv'),
    pair_configuration_sha256=file_digest(PAIR_DIR/'configuration.json'),
    simulator_configuration_sha256=file_digest(SIM_DIR/'configuration.json'),
    bank_directory=str(BANK_DIR),source33_configuration_sha256=source33_sha,
    temperature=.05,tail_fraction=.1,lambda_auxiliary=.1,triplet_margin=.2,negative_identities=7,
    matcher_thresholds=source_thresholds,
    checkpoint_every_updates=CHECKPOINT_EVERY,
    selection='same fixed72-source ten-condition patch global L1; include common original23 epoch0; strict improvement',
    inference='one full native normalized image, FP32, no AMP, no padding, no tiles',
    precision='FP32, deterministic; AMP/TF32 off',validation_is_not_independent_test=True,
    gpu_name=torch.cuda.get_device_name(0) if CHECK_DEVICE.type=='cuda' else None,
    gpu_compute_capability=list(torch.cuda.get_device_capability(0)) if CHECK_DEVICE.type=='cuda' else None,
    torch_version=str(torch.__version__),torch_cuda=str(torch.version.cuda),
    numpy_version=np.__version__,pillow_version=Image.__version__,
    sources={name:hashlib.sha256(text.encode()).hexdigest() for name,text in
        [('bank.py.txt',BANK_SOURCE),('training.py.txt',TRAIN_SOURCE),
         ('native_image.py.txt',IMAGE_SOURCE),('selector.py.txt',SELECTOR_SOURCE)]})
experiment_sha=hashlib.sha256(json_bytes(experiment_config)).hexdigest()
EXPERIMENT_DIR=PILOT_ROOT/experiment_sha[:20]
EXPERIMENT_DIR.mkdir(parents=True,exist_ok=True)
for name,data in {'configuration.json':json_bytes(experiment_config),
        'schedule.csv':schedule.to_csv(index=False,lineterminator='\n').encode(),
        'bank.py.txt':BANK_SOURCE.encode(),'training.py.txt':TRAIN_SOURCE.encode(),
        'native_image.py.txt':IMAGE_SOURCE.encode(),'selector.py.txt':SELECTOR_SOURCE.encode()}.items():
    path=EXPERIMENT_DIR/name
    if path.exists() and path.read_bytes()!=data:
        raise RuntimeError(f'Frozen pilot experiment differs: {path}')
    if not path.exists():
        write_verified(path,data)
print('TWO-ARM NATIVE REPLICATION:',EXPERIMENT_DIR,'| updates/arm:',experiment_config['updates_per_run'])


In [ ]:
results37={name:train_native_arm(name) for name in PILOT_RUNS}


In [ ]:

if state_sha(dmd_model)!=initial_dmd_sha or any(p.grad is not None for p in dmd_model.parameters()):
    raise RuntimeError('Frozen DMD weights/gradients changed.')
verify_files(SOURCE36_DIR,source36_completion['file_sha256'])
verify_files(SOURCE35_DIR,source35_completion['file_sha256'])
if file_digest(original_run/original_best['file'])!=original_best['sha256']:
    raise RuntimeError('Original23 warmstart changed.')
completion37=dict(status='native_counterfactual_pilot_complete',configuration_sha256=experiment_sha,
    epochs=PILOT_EPOCHS,run_names=list(PILOT_RUNS),
    run_completion_sha256={name:file_digest(EXPERIMENT_DIR/name/'completion.json') for name in PILOT_RUNS},
    file_sha256={name:file_digest(EXPERIMENT_DIR/name) for name in
        ('configuration.json','schedule.csv','bank.py.txt','training.py.txt','native_image.py.txt','selector.py.txt')},
    recognition_improvement_verified=False,external_test_evaluated=False,legacy30_31_authorized=False)
write_verified(EXPERIMENT_DIR/'completion.json',json_bytes(completion37))
write_verified(PILOT_ROOT/'completed_experiment.json',json_bytes(dict(directory=str(EXPERIMENT_DIR),
    configuration_sha256=experiment_sha,completion_sha256=file_digest(EXPERIMENT_DIR/'completion.json'))))
print(json.dumps(completion37,indent=2))
print('REPLICATION39 TRAINING COMPLETE:',EXPERIMENT_DIR)
print('Next: run notebook 40 for NBIS validation. Training losses are not recognition results.')
